<a href="https://colab.research.google.com/github/kuds/reinforce-tactics/blob/main/notebooks/feudal_rl_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Reinforce Tactics — Feudal RL Training

This notebook trains and validates the **Feudal RL** (Manager–Worker hierarchy) agent.
It mirrors the PPO notebook's evaluation setup so feudal and flat-PPO results
are directly comparable: same starter map, same enabled unit subset, same
``reward_config`` (Direction A — terminal-dominant, capture-aligned), and the
same SB3-style periodic eval cadence rather than a hand-picked checkpoint list.

## What is Feudal RL?

The Feudal RL architecture splits decision-making into two levels:

| Level | Network | Output | Update Frequency |
|-------|---------|--------|------------------|
| **Manager** | `ManagerNetwork` | Spatial goal `(x, y, type)` | Every `manager_horizon` steps |
| **Worker** | `WorkerNetwork` | Primitive action `(action_type, unit_type, from, to)` | Every step |

The **Manager** sets strategic goals (attack, defend, capture, expand) and the
**Worker** executes tactical actions conditioned on those goals. Both are trained
with PPO using separate advantage estimation.

**Runtime:** ~5–10 min on CPU, ~2–5 min on GPU (T4/L4).

---

## 1. Setup

In [ ]:
# Install dependencies
!pip install -q gymnasium stable-baselines3 sb3-contrib tensorboard numpy torch matplotlib

# Install reinforce-tactics from GitHub
!pip install -q git+https://github.com/kuds/reinforce-tactics.git

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
import time
from collections import defaultdict

import matplotlib.pyplot as plt
import numpy as np

from reinforcetactics.rl.evaluation import evaluate_model
from reinforcetactics.rl.feudal_rl import FeudalRLAgent, compute_intrinsic_reward
from reinforcetactics.rl.gym_env import StrategyGameEnv
from reinforcetactics.rl.viz import (
    plot_action_distribution,
    plot_eval_curves,
    plot_outcome_breakdown,
    plot_reward_decomposition,
)

print("Imports OK")

## 2. Configuration

Adjust these hyperparameters to experiment with training.

In [ ]:
# --- Training config ---
TOTAL_TIMESTEPS = 25_000  # Total environment steps
N_STEPS = 512  # Steps per rollout (smaller for notebook)
BATCH_SIZE = 64  # PPO mini-batch size
N_EPOCHS = 4  # PPO epochs per update
LEARNING_RATE = 3e-4  # Base learning rate
MANAGER_LR_SCALE = 1.0  # Manager LR = base * scale
WORKER_LR_SCALE = 1.0  # Worker LR = base * scale

# --- Feudal RL config ---
MANAGER_HORIZON = 10  # Worker steps between manager goal updates
WORKER_REWARD_ALPHA = 0.5  # Extrinsic reward weight (0=intrinsic only)
# Multiplier applied to extrinsic rewards before they hit the buffer.
# Direction-A REWARD_CONFIG below has terminal magnitudes of ±5000, which
# blows up the value-function MSE if fed in raw. 0.001 maps the dynamic
# range to roughly ±5 so vf_coef * value_loss stays in budget.
REWARD_SCALE = 0.001
# Worker head: legacy 6-head independent Categoricals (False) vs.
# AlphaStar-style autoregressive head with exact per-stage masking (True).
# AR pairs with env.structured_action_masks() so each sampling stage sees
# only legal options conditional on prior stages, eliminating the
# over-approximation of the per-dimension product mask.
USE_AUTOREGRESSIVE_WORKER = False

# --- PPO config ---
GAMMA = 0.99
GAE_LAMBDA = 0.95
CLIP_RANGE = 0.2
ENT_COEF = 0.05
VF_COEF = 0.5
MAX_GRAD_NORM = 0.5

# --- Environment config (mirrors PPO notebook for direct comparison) ---
MAP_FILE = "maps/1v1/starter.csv"  # 4x4 playable area inside 6x6 ocean borders
OPPONENT = "random"  # Primary training opponent (eval also runs vs other bots)
MAX_STEPS = 400  # Max env steps per episode (raised so MAX_TURNS triggers first)
MAX_TURNS = 20  # Max game turns before auto-draw (terminated, not truncated)

# Enabled unit subset — keeps the action space smaller so the pipeline can be
# validated before opening up the full roster. W = Warrior, M = Mage, A = Archer.
ENABLED_UNITS = ["W", "M", "A"]

# Feudal worker uses multi-discrete actions; keep that mode here.
ACTION_SPACE = "multi_discrete"

# --- Reward configuration (Direction A — terminal-dominant, capture-aligned) ---
# Same shape as the PPO notebook so feudal and flat-PPO results are
# directly comparable. Win/loss/draw dominate per-step shaping, draw == loss,
# structure_control + seize_progress + capture point the gradient at the HQ.
REWARD_CONFIG = {
    "win": 5000.0,
    "loss": -5000.0,
    "draw": -5000.0,
    "income_diff": 0.5,
    "unit_diff": 1.0,
    "structure_control": 10.0,
    "create_unit": 0.0,
    "move": 0.0,
    "damage_scale": 0.0,
    "kill": 0.0,
    "seize_progress": 10.0,
    "capture": 1000.0,
    "cure": 0.0,
    "heal_scale": 0.0,
    "paralyze": 0.0,
    "haste": 0.0,
    "defence_buff": 0.0,
    "attack_buff": 0.0,
    "invalid_action": -10.0,
    "turn_penalty": -20.0,
}

# --- Eval config (SB3 EvalCallback-style: fixed cadence, not hand-picked) ---
EVAL_FREQ = 5_000  # Run an evaluation every N env steps
N_EVAL_EPISODES = 10  # Episodes per evaluation per opponent
EVAL_OPPONENTS = ["random", "bot"]  # Multi-opponent eval — random is the train target, bot is harder

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("Config loaded")
print(f"  Map:          {MAP_FILE}")
print(f"  Opponent:     {OPPONENT}")
print(f"  Eval vs:      {EVAL_OPPONENTS}")
print(f"  Enabled units:{ENABLED_UNITS}")
print(f"  Max steps:    {MAX_STEPS}")
print(f"  Max turns:    {MAX_TURNS}")
print(f"  Total steps:  {TOTAL_TIMESTEPS:,}")
print(f"  Eval freq:    {EVAL_FREQ:,} env steps  ({TOTAL_TIMESTEPS // EVAL_FREQ} evaluations)")
print(f"  Manager horizon: {MANAGER_HORIZON}")
print(f"  Worker reward alpha: {WORKER_REWARD_ALPHA}")
print(f"  Reward scale:        {REWARD_SCALE}")
print(f"  Autoregressive worker: {USE_AUTOREGRESSIVE_WORKER}")

## 3. Environment & Agent Setup

In [ ]:
# Training env (single — feudal's collect_rollout is single-env).
env = StrategyGameEnv(
    map_file=MAP_FILE,
    opponent=OPPONENT,
    render_mode=None,
    max_steps=MAX_STEPS,
    max_turns=MAX_TURNS,
    reward_config=REWARD_CONFIG,
    enabled_units=ENABLED_UNITS,
    action_space_type=ACTION_SPACE,
)

# One eval env per opponent. Each is reset with seed=SEED+ep_idx by
# evaluate_model so results are reproducible across runs.
eval_envs = {
    opp: StrategyGameEnv(
        map_file=MAP_FILE,
        opponent=opp,
        render_mode=None,
        max_steps=MAX_STEPS,
        max_turns=MAX_TURNS,
        reward_config=REWARD_CONFIG,
        enabled_units=ENABLED_UNITS,
        action_space_type=ACTION_SPACE,
    )
    for opp in EVAL_OPPONENTS
}
# Primary eval env (used by sanity checks + watch-the-agent cell).
eval_env = eval_envs[OPPONENT]

print(f"Grid: {env.grid_width}x{env.grid_height}")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print(f"Eval envs: {list(eval_envs.keys())}")

In [ ]:
# Create the Feudal RL agent
agent = FeudalRLAgent(
    observation_space=env.observation_space,
    grid_width=env.grid_width,
    grid_height=env.grid_height,
    agent_player=getattr(env, "agent_player", 1),
    device=DEVICE,
    autoregressive_worker=USE_AUTOREGRESSIVE_WORKER,
)
agent.manager_horizon = MANAGER_HORIZON

# Setup training optimizers
agent.setup_training(
    learning_rate=LEARNING_RATE,
    manager_lr_scale=MANAGER_LR_SCALE,
    worker_lr_scale=WORKER_LR_SCALE,
)

# collect_rollout auto-initializes _last_obs on first call, so we just need
# to clear any stale goal state before the first rollout.
agent.reset_goal()


# Count parameters
def count_params(model):
    return sum(p.numel() for p in model.parameters())


print("\nModel parameters:")
print(f"  Feature extractor: {count_params(agent.feature_extractor):,}")
print(f"  Manager:           {count_params(agent.manager):,}")
print(f"  Worker:            {count_params(agent.worker):,}  ({'AR' if USE_AUTOREGRESSIVE_WORKER else '6-head'})")
total = count_params(agent.feature_extractor) + count_params(agent.manager) + count_params(agent.worker)
print(f"  Total:             {total:,}")

## 4. Sanity Checks

Before training, verify the pipeline works with a single rollout + update.

In [ ]:
print("Running sanity checks...\n")

# 1. Test select_action
test_obs = env.reset(seed=0)[0]
action, goal = agent.select_action(test_obs)
print(f"[OK] select_action -> action={action}, goal={goal}")
assert action.shape == (6,), f"Bad action shape: {action.shape}"
assert goal.shape == (3,), f"Bad goal shape: {goal.shape}"

# Reset for training. collect_rollout will auto-init _last_obs from env.reset()
# on the first call, but we still call reset_goal() so the manager starts fresh.
env.reset(seed=SEED)
agent._last_obs = None
agent.reset_goal()

# 2. Test collect_rollout
buf = agent.collect_rollout(
    env,
    n_steps=64,
    gamma=GAMMA,
    gae_lambda=GAE_LAMBDA,
    worker_reward_alpha=WORKER_REWARD_ALPHA,
    reward_scale=REWARD_SCALE,
)
print(f"[OK] collect_rollout -> {len(buf.w_rewards)} worker steps, {len(buf.m_rewards)} manager segments")
assert buf.w_advantages is not None
assert len(buf.w_rewards) == 64

# 3. Test update
metrics = agent.update(
    buf, n_epochs=2, batch_size=32, clip_range=CLIP_RANGE, ent_coef=ENT_COEF, vf_coef=VF_COEF, max_grad_norm=MAX_GRAD_NORM
)
print("[OK] update -> losses: ", end="")
for k, v in metrics.items():
    print(f"{k}={v:.4f} ", end="")
    assert np.isfinite(v), f"{k} is not finite: {v}"
print()

# 4. Test intrinsic reward (signature dropped the unused `state` and
# `agent_player` args — the function reads `self`/`opp` directly off the
# agent-relative observation channels). Build the test_state to match the
# real observation_space shape rather than hardcoded sizes so this stays
# correct if the channel layout in rl.observation evolves.
from reinforcetactics.rl.observation import NUM_UNIT_TYPES as _NUM_UT

ut_shape = env.observation_space["units"].shape
gd_shape = env.observation_space["grid"].shape
gf_shape = env.observation_space["global_features"].shape
test_state = {
    "grid": np.zeros(gd_shape, dtype=np.float32),
    "units": np.zeros(ut_shape, dtype=np.float32),
    "global_features": np.zeros(gf_shape, dtype=np.float32),
}
test_state["units"][5, 5, _NUM_UT + 0] = 1  # player unit on the "self" owner channel
test_goal = np.array([5, 5, 0])  # attack goal at unit position
ir = compute_intrinsic_reward(test_state, test_goal)
print(f"[OK] intrinsic_reward -> {ir:.2f} (unit at goal)")
assert ir >= 5.0, f"Expected >= 5.0 for unit at goal, got {ir}"

# 5. Test evaluate
eval_results = agent.evaluate(eval_env, n_episodes=3)
print(f"[OK] evaluate -> reward={eval_results['mean_reward']:.1f}, win_rate={eval_results['win_rate']:.2f}")

print("\nAll sanity checks passed!")

## 5. Training Loop

Train the feudal agent and record metrics at checkpoints.

In [ ]:
# Reset environment and agent for fresh training. collect_rollout will
# auto-init the rolling _last_obs from env.reset() on its first call.
env.reset(seed=SEED)
agent._last_obs = None
agent.reset_goal()


# Adapter so feudal's select_action() can be passed to evaluate_model().
# evaluate_model concatenates per-dim masks into a flat MaskablePPO-style
# array; feudal needs the original 6-tuple, so we fetch masks directly
# from the env inside predict() and ignore the kwarg evaluate_model passes.
# In AR mode we instead fetch env.structured_action_masks() and pass it via
# the new structured_masks= kwarg so each AR stage masks against the exact
# legal-set conditional on prior stages.
class FeudalPredictShim:
    def __init__(self, agent, env):
        self.agent = agent
        self.env = env

    def predict(self, obs, deterministic=True, action_masks=None, state=None, episode_start=None):
        if self.agent.autoregressive_worker and hasattr(self.env, "structured_action_masks"):
            sm = self.env.structured_action_masks()
            action, _goal = self.agent.select_action(obs, deterministic=deterministic, structured_masks=sm)
        else:
            masks = self.env.action_masks() if hasattr(self.env, "action_masks") else None
            action, _goal = self.agent.select_action(obs, deterministic=deterministic, action_masks=masks)
        return action, state


# evaluate_model has no per-episode hook the shim could use to call
# agent.reset_goal(). Without this, the first ~MANAGER_HORIZON steps of every
# eval episode would run on the previous episode's goal. We also hide
# action_masks from evaluate_model so it skips the MaskablePPO concat path —
# the shim fetches masks itself in the multi_discrete tuple form feudal expects.
class _GoalResettingEnv:
    def __init__(self, env, agent):
        self._env = env
        self._agent = agent

    def reset(self, **kwargs):
        self._agent.reset_goal()
        return self._env.reset(**kwargs)

    def step(self, action):
        return self._env.step(action)

    def __getattr__(self, name):
        # Hide action_masks from evaluate_model — the shim fetches them
        # directly via the underlying env reference.
        if name == "action_masks":
            raise AttributeError(name)
        return getattr(self._env, name)


def eval_against(opponent_label):
    """Run evaluate_model against the named opponent's eval_env, then reset
    the agent's goal so the next training rollout starts cleanly."""
    raw_env = eval_envs[opponent_label]
    target_env = _GoalResettingEnv(raw_env, agent)
    shim = FeudalPredictShim(agent, raw_env)  # shim fetches masks from raw env
    res = evaluate_model(
        shim,
        target_env,
        n_episodes=N_EVAL_EPISODES,
        deterministic=True,
        seed=SEED,
        track_breakdown=True,
    )
    agent.reset_goal()
    return res


# Per-opponent result lists (each shaped like the PPO notebook's eval_callback.results).
results_per_opp = {opp: [] for opp in EVAL_OPPONENTS}
best_win_rate = -1.0
best_step = 0

# Training-internals tracking (feudal-specific; PPO's TrainingMetricsCallback equivalent).
history = defaultdict(list)
num_updates = TOTAL_TIMESTEPS // N_STEPS
total_timesteps = 0
last_eval_step = 0
start_time = time.time()

print(f"Training for {TOTAL_TIMESTEPS:,} timesteps ({num_updates} updates)")
print(f"Eval every {EVAL_FREQ:,} env steps vs {EVAL_OPPONENTS}")
print(f"{'=' * 60}")

for update_idx in range(num_updates):
    # Collect rollout
    buf = agent.collect_rollout(
        env,
        n_steps=N_STEPS,
        gamma=GAMMA,
        gae_lambda=GAE_LAMBDA,
        worker_reward_alpha=WORKER_REWARD_ALPHA,
        reward_scale=REWARD_SCALE,
    )
    total_timesteps += N_STEPS

    # PPO update
    metrics = agent.update(
        buf,
        n_epochs=N_EPOCHS,
        batch_size=BATCH_SIZE,
        clip_range=CLIP_RANGE,
        ent_coef=ENT_COEF,
        vf_coef=VF_COEF,
        max_grad_norm=MAX_GRAD_NORM,
    )

    # Record training-internals metrics
    history["timestep"].append(total_timesteps)
    for key, val in metrics.items():
        history[key].append(val)
    history["worker_mean_reward"].append(float(buf.w_rewards.mean()))
    history["worker_std_reward"].append(float(buf.w_rewards.std()))
    history["manager_segments"].append(len(buf.m_rewards))
    history["episode_dones"].append(float(buf.w_dones.sum()))

    # Pass-2: surface env info from the rollout (end_reasons, reward_breakdown).
    for reason in getattr(buf, "end_reasons", []):
        history.setdefault("rollout_end_reasons", []).append(reason)
    for k, v in getattr(buf, "reward_breakdown", {}).items():
        history[f"rollout_reward_{k}"] = history.get(f"rollout_reward_{k}", 0.0) + float(v)

    if buf.has_manager_data:
        goal_types = buf.m_goals[:, 2].astype(int)
        for gt in range(4):
            history[f"goal_type_{gt}_pct"].append((goal_types == gt).sum() / max(len(goal_types), 1))
    else:
        for gt in range(4):
            history[f"goal_type_{gt}_pct"].append(0.0)

    if (update_idx + 1) % 5 == 0:
        elapsed = time.time() - start_time
        sps = total_timesteps / elapsed
        # Recent end-reason mix from the last rollout (pass-2 diagnostic).
        recent_reasons = getattr(buf, "end_reasons", [])
        reason_str = ",".join(recent_reasons) if recent_reasons else "—"
        print(
            f"[{total_timesteps:>7,}] "
            f"w_policy={metrics['worker_policy_loss']:.3f} "
            f"m_policy={metrics['manager_policy_loss']:.3f} "
            f"w_entropy={metrics['worker_entropy']:.3f} "
            f"w_reward={buf.w_rewards.mean():.2f} "
            f"end={reason_str} "
            f"({sps:.0f} steps/s)"
        )

    # Periodic eval gate (SB3 EvalCallback-style; replaces hand-picked CHECKPOINT_STEPS).
    if total_timesteps - last_eval_step >= EVAL_FREQ or update_idx == num_updates - 1:
        last_eval_step = total_timesteps
        print(f"  >> EVAL @ {total_timesteps:,}:")
        for opp in EVAL_OPPONENTS:
            res = eval_against(opp)
            res["timesteps"] = total_timesteps
            results_per_opp[opp].append(res)
            print(
                f"     vs {opp:8s}: win_rate={res['win_rate']:.2f}  "
                f"reward={res['avg_reward']:+.1f} +/- {res['std_reward']:.1f}  "
                f"len={res['avg_length']:.0f}  "
                f"W/L/D={res['wins']}/{res['losses']}/{res['draws']}"
            )

        # Best-checkpoint tracking — keyed on the primary training opponent
        # so we save the model that's strongest at the task it's training on.
        primary_wr = results_per_opp[OPPONENT][-1]["win_rate"]
        if primary_wr > best_win_rate:
            best_win_rate = primary_wr
            best_step = total_timesteps
            agent.save_checkpoint("/tmp/feudal_best.pt")
            print(f"     [BEST] win_rate={best_win_rate:.2f} @ {best_step:,}")

elapsed = time.time() - start_time
print(f"{'=' * 60}")
print(f"Training complete in {elapsed:.1f}s ({total_timesteps / elapsed:.0f} steps/s)")
print(f"Best win rate vs {OPPONENT}: {best_win_rate:.2f} @ {best_step:,} steps")

# Convenience alias — the PPO notebook plots from a single ``results`` list.
results = results_per_opp[OPPONENT]

## 6. Results & Visualization

In [ ]:
# Per-opponent eval table.
for opp in EVAL_OPPONENTS:
    rs = results_per_opp[opp]
    if not rs:
        continue
    print(f"\n=== Eval vs {opp} ===")
    print(f"{'Timesteps':>12} {'Avg Reward':>12} {'Std Reward':>12} {'Win Rate':>10} {'Len':>6} {'W/L/D':>9}")
    print("-" * 70)
    for r in rs:
        wld = f"{r['wins']}/{r['losses']}/{r['draws']}"
        print(
            f"{r['timesteps']:>12,} "
            f"{r['avg_reward']:>12.1f} "
            f"{r['std_reward']:>12.1f} "
            f"{r['win_rate']:>10.2f} "
            f"{r['avg_length']:>6.0f} "
            f"{wld:>9}"
        )

print(f"\nBest win rate vs {OPPONENT}: {best_win_rate:.2f} @ {best_step:,} steps")

# Pass-2: rollout-time end-reason mix and accumulated reward components.
# These come from info[end_reason] / info[reward_breakdown] in the env, which
# the rollout loop now surfaces on the buffer. Useful for catching things
# like "agent only ever truncates" or "shaping_delta dominates terminal".
rollout_reasons = history.get("rollout_end_reasons", [])
if rollout_reasons:
    from collections import Counter

    print("\nRollout end-reason distribution (across all training episodes):")
    for reason, count in Counter(rollout_reasons).most_common():
        pct = count / len(rollout_reasons) * 100
        print(f"  {reason:24s} {count:5d}  ({pct:.0f}%)")

rollout_components = {k.removeprefix("rollout_reward_"): v for k, v in history.items() if k.startswith("rollout_reward_")}
if rollout_components:
    print("\nRollout reward-component sums (across all training steps):")
    for k, v in sorted(rollout_components.items(), key=lambda kv: -abs(kv[1])):
        print(f"  {k:18s} {v:+12.1f}")

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.suptitle("Feudal RL Training Metrics", fontsize=14)

timesteps = history["timestep"]

# Worker policy loss
axes[0, 0].plot(timesteps, history["worker_policy_loss"], alpha=0.7)
axes[0, 0].set_title("Worker Policy Loss")
axes[0, 0].set_xlabel("Timesteps")
axes[0, 0].grid(True, alpha=0.3)

# Manager policy loss
axes[0, 1].plot(timesteps, history["manager_policy_loss"], alpha=0.7, color="orange")
axes[0, 1].set_title("Manager Policy Loss")
axes[0, 1].set_xlabel("Timesteps")
axes[0, 1].grid(True, alpha=0.3)

# Entropy
axes[0, 2].plot(timesteps, history["worker_entropy"], label="Worker", alpha=0.7)
axes[0, 2].plot(timesteps, history["manager_entropy"], label="Manager", alpha=0.7)
axes[0, 2].set_title("Entropy")
axes[0, 2].set_xlabel("Timesteps")
axes[0, 2].legend()
axes[0, 2].grid(True, alpha=0.3)

# Gradient norms (pre-clip) — flag exploding/vanishing gradients early.
if "worker_grad_norm" in history:
    axes[0, 3].plot(timesteps, history["worker_grad_norm"], label="Worker", alpha=0.7)
    axes[0, 3].plot(timesteps, history["manager_grad_norm"], label="Manager", alpha=0.7)
    axes[0, 3].set_title("Gradient Norm (pre-clip)")
    axes[0, 3].set_xlabel("Timesteps")
    axes[0, 3].set_yscale("log")
    axes[0, 3].legend()
    axes[0, 3].grid(True, alpha=0.3)
else:
    axes[0, 3].axis("off")

# Worker reward
axes[1, 0].plot(timesteps, history["worker_mean_reward"], alpha=0.7, color="green")
axes[1, 0].fill_between(
    timesteps,
    np.array(history["worker_mean_reward"]) - np.array(history["worker_std_reward"]),
    np.array(history["worker_mean_reward"]) + np.array(history["worker_std_reward"]),
    alpha=0.15,
    color="green",
)
axes[1, 0].set_title("Worker Mean Reward (per step)")
axes[1, 0].set_xlabel("Timesteps")
axes[1, 0].grid(True, alpha=0.3)

# Value losses
axes[1, 1].plot(timesteps, history["worker_value_loss"], label="Worker", alpha=0.7)
axes[1, 1].plot(timesteps, history["manager_value_loss"], label="Manager", alpha=0.7)
axes[1, 1].set_title("Value Loss")
axes[1, 1].set_xlabel("Timesteps")
axes[1, 1].set_yscale("log")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# Goal type distribution
goal_labels = ["Attack", "Defend", "Capture", "Expand"]
for gt in range(4):
    axes[1, 2].plot(timesteps, history[f"goal_type_{gt}_pct"], label=goal_labels[gt], alpha=0.7)
axes[1, 2].set_title("Manager Goal Distribution")
axes[1, 2].set_xlabel("Timesteps")
axes[1, 2].set_ylabel("Fraction")
axes[1, 2].legend()
axes[1, 2].grid(True, alpha=0.3)

# Manager segments per rollout
axes[1, 3].plot(timesteps, history["manager_segments"], alpha=0.7, color="purple")
axes[1, 3].set_title("Manager Segments / Rollout")
axes[1, 3].set_xlabel("Timesteps")
axes[1, 3].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Shared viz plotters (same as PPO notebook). Per-opponent so feudal vs
# random and feudal vs bot are immediately comparable side by side.
for opp in EVAL_OPPONENTS:
    rs = results_per_opp[opp]
    if not rs:
        continue
    print(f"\n--- Eval curves vs {opp} ---")
    fig = plot_eval_curves(rs, opponent_label=opp)
    plt.show()

# Reward-component decomposition / action distribution / outcome × end-reason
# breakdown — only meaningful with track_breakdown=True (set above).
print("\n--- Reward decomposition (vs primary opponent) ---")
fig = plot_reward_decomposition(results)
if fig is not None:
    plt.show()

print("\n--- Action distribution (vs primary opponent) ---")
fig = plot_action_distribution(results)
if fig is not None:
    plt.show()

print("\n--- Outcome × end-reason breakdown (vs primary opponent) ---")
fig = plot_outcome_breakdown(results)
if fig is not None:
    plt.show()

## 7. Pipeline Validation

Verify that the training pipeline produces expected outputs and the model
can be saved/loaded correctly.

In [ ]:
import tempfile
from pathlib import Path

print("Running validation checks...\n")
all_passed = True

# 1. Training produced finite losses
for key in ["worker_policy_loss", "manager_policy_loss", "worker_entropy"]:
    vals = history[key]
    if all(np.isfinite(v) for v in vals):
        print(f"  [PASS] {key}: all finite (last={vals[-1]:.4f})")
    else:
        print(f"  [FAIL] {key}: contains non-finite values")
        all_passed = False

# 2. Worker entropy stays positive (exploration)
if all(v > 0 for v in history["worker_entropy"]):
    print(f"  [PASS] Worker entropy always positive (min={min(history['worker_entropy']):.4f})")
else:
    print("  [FAIL] Worker entropy went to zero")
    all_passed = False

# 3. Manager segments were created
total_segments = sum(history["manager_segments"])
if total_segments > 0:
    print(f"  [PASS] Manager produced {total_segments} goal segments")
else:
    print("  [FAIL] No manager segments created")
    all_passed = False

# 4. Goal type diversity
last_goal_pcts = [history[f"goal_type_{gt}_pct"][-1] for gt in range(4)]
nonzero_types = sum(1 for p in last_goal_pcts if p > 0)
if nonzero_types >= 2:
    print(f"  [PASS] Manager uses {nonzero_types}/4 goal types (diverse goals)")
else:
    print(f"  [WARN] Manager only uses {nonzero_types}/4 goal types (may need more training)")

# 5. Checkpoint save/load roundtrip — also verify saved hyperparams
# (manager_horizon, agent_player) survive the round trip without the
# caller having to re-set them.
with tempfile.TemporaryDirectory() as tmpdir:
    ckpt_path = str(Path(tmpdir) / "feudal_test.pt")
    agent.save_checkpoint(ckpt_path)

    # Reconstruct with the matching worker-head flag — load_checkpoint refuses
    # to load a state_dict from a different head shape (or different grid dims).
    agent2 = FeudalRLAgent(
        observation_space=env.observation_space,
        grid_width=env.grid_width,
        grid_height=env.grid_height,
        device=DEVICE,
        autoregressive_worker=USE_AUTOREGRESSIVE_WORKER,
    )
    # Default manager_horizon is 10; if saved value differs, load should restore it.
    pre_load_horizon = agent2.manager_horizon
    agent2.load_checkpoint(ckpt_path)

    # Verify weights match
    match = all(torch.allclose(p1, p2) for p1, p2 in zip(agent.manager.parameters(), agent2.manager.parameters()))
    if match:
        print("  [PASS] Checkpoint save/load roundtrip")
    else:
        print("  [FAIL] Checkpoint weights don't match after load")
        all_passed = False

    if agent2.manager_horizon == agent.manager_horizon:
        print(f"  [PASS] manager_horizon restored from checkpoint ({pre_load_horizon} -> {agent2.manager_horizon})")
    else:
        print(f"  [FAIL] manager_horizon not restored: got {agent2.manager_horizon}, expected {agent.manager_horizon}")
        all_passed = False

    # Verify loaded model can act
    test_obs = env.reset(seed=0)[0]
    action, goal = agent2.select_action(test_obs)
    if action.shape == (6,):
        print("  [PASS] Loaded model produces valid actions")
    else:
        print(f"  [FAIL] Loaded model action shape: {action.shape}")
        all_passed = False

# 6. Episodes completed during training
total_dones = sum(history["episode_dones"])
if total_dones > 0:
    print(f"  [PASS] Completed {int(total_dones)} episodes during training")
else:
    print("  [WARN] No episodes completed (may need longer training or shorter episodes)")

print()
if all_passed:
    print("All validation checks PASSED!")
else:
    print("Some validation checks FAILED - review output above.")

## 8. Watch the Agent Play

Run a single game and display the manager's goal decisions.

In [ ]:
goal_names = {0: "Attack", 1: "Defend", 2: "Capture", 3: "Expand"}
action_names = {
    0: "Create",
    1: "Move",
    2: "Attack",
    3: "Seize",
    4: "Heal",
    5: "EndTurn",
    6: "Paralyze",
    7: "Haste",
    8: "DefBuff",
    9: "AtkBuff",
}

# Watch env must match the training config — otherwise the grid size differs
# from what the feature extractor was trained on and the linear projection
# matmul will mismatch.
watch_env = StrategyGameEnv(
    map_file=MAP_FILE,
    opponent=OPPONENT,
    render_mode=None,
    max_steps=MAX_STEPS,
    max_turns=MAX_TURNS,
    reward_config=REWARD_CONFIG,
    enabled_units=ENABLED_UNITS,
    action_space_type=ACTION_SPACE,
)
obs, _ = watch_env.reset(seed=99)
agent.reset_goal()

done = False
step = 0
ep_reward = 0.0
goal_history = []

while not done and step < 100:  # Cap display at 100 steps
    if agent.autoregressive_worker:
        sm = watch_env.structured_action_masks()
        action, goal = agent.select_action(obs, deterministic=True, structured_masks=sm)
    else:
        masks = watch_env.action_masks() if hasattr(watch_env, "action_masks") else None
        action, goal = agent.select_action(obs, deterministic=True, action_masks=masks)
    obs, reward, terminated, truncated, info = watch_env.step(action)
    done = terminated or truncated
    ep_reward += reward
    step += 1

    goal_type = int(goal[2])
    action_type = int(action[0])
    goal_history.append(goal_type)

    if step <= 20 or done:  # Show first 20 steps + final
        valid = info.get("valid_action", "?")
        print(
            f"  Step {step:3d} | "
            f"Goal: {goal_names.get(goal_type, '?'):8s} ({int(goal[0]):2d},{int(goal[1]):2d}) | "
            f"Action: {action_names.get(action_type, '?'):8s} | "
            f"Valid: {valid} | Reward: {reward:+.1f}"
        )

if step > 20 and not done:
    print(f"  ... ({step - 20} more steps)")

print(f"\nGame ended at step {step}. Total reward: {ep_reward:.1f}")
winner = info.get("winner")
if winner == 1:
    print("Result: Agent WON")
elif winner is not None:
    print(f"Result: Agent LOST (winner: player {winner})")
else:
    print("Result: Draw / Truncated")

end_reason = info.get("end_reason")
if end_reason:
    print(f"End reason: {end_reason}")

# Goal type usage summary
print("\nGoal type distribution across game:")
for gt in range(4):
    count = sum(1 for g in goal_history if g == gt)
    pct = count / max(len(goal_history), 1) * 100
    print(f"  {goal_names[gt]:8s}: {count:3d} ({pct:.0f}%)")

## 9. Next Steps

This notebook validated the feudal RL pipeline end-to-end with the same
env config / reward shape / eval cadence as the PPO notebook, so feudal
and flat-PPO results on `maps/1v1/starter.csv` are directly comparable.

**Pass (1) parity covered here:**
- Starter map + W/M/A unit subset, `max_turns=20`, Direction-A `reward_config`
- SB3-style periodic eval at fixed cadence (replaces hand-picked `CHECKPOINT_STEPS`)
- Multi-opponent eval (`random`, `bot`) via `evaluate_model` + a `predict()` shim
- Best-checkpoint tracking (saved to `/tmp/feudal_best.pt`)
- Shared `viz.py` plotters: eval curves, reward decomposition, action distribution, outcome × end-reason

**Pass (2) — applied (changes feudal's learning behavior):**
- Worker logits are masked using `env.action_masks()` at sample, eval, and update time
  (per-dim masks aligned with `[action_type, unit_type, from_x, from_y, to_x, to_y]`)
- Per-dim masks are stored in the rollout buffer so PPO's importance-sampling ratio
  remains well-defined under masking
- `info["end_reason"]` and `info["reward_breakdown"]` are surfaced from
  `collect_rollout` onto the buffer (`buf.end_reasons`, `buf.reward_breakdown`)
  and shown in training progress + the rollout-summary cell

For full training:

1. **Scale up**: Use `scripts/train/train_feudal_rl.py` with `--total-timesteps 10000000`
2. **Tune the manager horizon**: Try `--manager-horizon 5` (faster goals) or `20` (longer-term strategy)
3. **Adjust reward balance**: `--worker-reward-alpha 0.3` emphasizes intrinsic rewards
4. **Compare with flat PPO**: Run `--mode flat --use-action-masking` for a baseline

```bash
python scripts/train/train_feudal_rl.py \
    --mode feudal \
    --total-timesteps 5000000 \
    --manager-horizon 10 \
    --worker-reward-alpha 0.5 \
    --opponent random \
    --device auto
```